<a href="https://colab.research.google.com/github/isc-patrick/python_for_devs/blob/main/Pandas_and_some_Jupyter_tricks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas Introduction

**A Pandas Dataframe is an in-memory, column oriented table**

Some good points to remember:
 - The Pandas library is built around one primary data structure, the Dataframe
 - A Dataframe is a column oriented, in-memory table
 - Manipulation is done via set operations similar to SQL, but it was built for matrix manipulation so the approach is slightly different

This notebook is broken into just three sections.
 1. Getting started
  1. Loading data and some simple usage
 2. Basic Syntax
 3. Additional Concepts


__Jupyter__
Throughout the notebook shortcuts and helper functions are introduced to get you comfortable using Jupyter. I provide a list of the few commands I use the most often below. First press CTRL+Shift+P and the command box will come up showing the command on the left and the keyboard shortcut on the right.

This is the list of keyboard shortcuts I use most often: (For Mac users, CTRL is replaced with CMD)

1. CTRL+Enter: Run current cell
2. Shift+Enter: Run current cell and move to next cell
3. Up/Down arrows to move from cell to cell
4. Clear all outputs. Later we will add a shotcut for this command


__Other Concepts__
2. inplace=False as default
  1. to avoid side-effects
  2. doesn't allow method chaining
3. Index is the primary key. There is only one index.
4. Understanding axis - axis=1=columns - can be confusing




# 1. Getting Started

In [ ]:
# An easy way to start learning about Dataframes is to use the read_csv method to load data
# As a first example you can use a file included in all Colab notebooks
# IF you had a connection to IRIS, you could call .read_sql(connection, sql)
import pandas as pd
path = '/content/sample_data/california_housing_test.csv'
df = pd.read_csv(path)

# Run these and then go to Edit in the main menu and select Clear all Outputs.

# .info() provides metadata for the dataframe
df.info()

# .head() shows the first n records. n is not required and defaults to 5
df.head()

# Look at the data. It represents aggregate data over some area that has some relation to lat and long.

In [ ]:
# Each column is a Series data structure and can be returned by proving the column name within []
# DataFrames and Series have a default index
population = df['population']
print(type(population))

# Use intellisense to review the properties and methods of a Series by adding a . after population and scrolling
# through the dropdown. You will see many obvious ones like min, max, mean, and a lot of others
# The wrench icon indicates a property and the box icon a method. Remember the last variable in a Jupyter cell is automatically printed, which you can test by running this cell as is.

# EXERCISE
population

In [55]:
# We can use this to get a nice html fotmatted dataframe
from IPython.display import display, HTML

# Here are some examples of manipulating the Dataframe
# These are meant just to see some working code. The next section, Basic Concepts, provides
# important information about how Pandas works generally so don't worry about syntax that is not clear right now.

# Select just some columns for the dataframe
cols_to_select = ['total_rooms', 'total_bedrooms', 'median_house_value']
df[cols_to_select]

def pretty_print_dataframe(df, msg = "", max_rows=10):
  if max_rows is None:
    max_rows = len(df)

  if msg:
    print(msg)

  if not isinstance(df, pd.DataFrame):
    df = pd.DataFrame(df)
  display(HTML(df[:max_rows].to_html()))

# Display the first 10 records
pretty_print_dataframe(df[cols_to_select], "First 5 records and 2 cols", 5)

# Now press CTRL+Shift(CMD)+P and type in Clear and select clear all outputs

First 5 records and 2 cols


,total_rooms,total_bedrooms,median_house_value
0,3885.0,661.0,344700.0
1,1510.0,310.0,176500.0
2,3589.0,507.0,270500.0
3,67.0,15.0,330000.0
4,1241.0,244.0,81700.0


In [ ]:
# Create a new column that divides the median house value by the total rooms
df['cost_per_room'] = df['median_house_value'] / df['total_rooms']
cols_to_select.append('cost_per_room')
df[cols_to_select]

# Select Tools from the main menu and sselect Keyboard shortcuts, about 7 down on the left side is clear all outputs
# This is the most useful shortcut that I use, but never has a default. Click into the cell next to Clear all Outputs and press CTRL+Shift+O or CMD+Shift+O and now that keyboard shortcut is set

# Ok, the final step is to go to Scroll up to the Getting Started header and collapse all the headers by clicking the arrows to the left of the titles. This is a really good way of keeping notebooks organized and making navigation quick and easy. Once all are collapsed, open the Basic Syntax section

# Basic Syntax

In [ ]:
'''
Filtering rows and columns

Dataframe syntax takes after R's data.frame syntax, which borrows from matrix syntax generally.
In this cell, I'm going to show examples that do the same thing with different syntax
so that you get a sense of the variation and can recognize different forms. At the end of this
section I am going to recommend using specific syntax that provides the most robust functionality.

This is all you need for single table access: SELECT <fields> FROM <dataframe> WHERE <conditions
'''

# Single column access
df.total_rooms
df['total_rooms']

# .loc takes 2 args, the first determines the rows, the second the columns
pretty_print_dataframe(df.loc[:10, 'total_rooms'])

# You can use method chaining as well
df[:10]['total_rooms']

# [] syntax just calls the classes __getitem__() method
# The bracket syntax exists because this familar syntax for people used to doing matrix computation

# Multiple column access
df[['total_rooms', 'total_bedrooms']]
df.loc[:, ['total_rooms', 'total_bedrooms']]

# Filtering out rows
df[df.total_rooms > 1000]
df.loc[df.total_rooms > 1000]
df.loc[df['total_rooms'] > 1000]

# Filtering out rows and columns. A boolean expression can also be provided to determine columns(not shown here)
df[df.total_rooms > 1000][['total_rooms', 'total_bedrooms']]
df.loc[df.total_rooms > 1000, ['total_rooms', 'total_bedrooms']]



In [ ]:
'''
Adding, removing and renaming columns
'''
new_column = 'cost_per_room'

# We already saw how to add a column
df[new_column] = df['median_house_value'] / df['total_rooms']


# Remove 'cost_per_room' (returns a new DataFrame)
df_new = df.drop(new_column, axis=1)

# Remove 'cost_per_room' (modifies the original DataFrame)
df.drop(new_column, axis=1, inplace=True)

# Rename a column
df = df.rename(columns={'total_rooms': 'rooms'})


In [57]:
'''
join, merge and concatenating dataframes
'''
import pandas as pd

# Patient Data
patient_columns = ['id', 'name', 'birthdate']
patients = pd.DataFrame([
    [1, 'Alice Smith', '1990-05-01'],
    [2, 'Bob Jones', '1985-07-13'],
    [3, 'Charlie Kim', '1975-12-22'],
    [4, 'Dana Lee', '1992-09-03'],
    [5, 'Eli Brown', '2000-01-30']
], columns=patient_columns)

# Encounter Data
encounters_columns = ['id', 'patient_id', 'date', 'notes']
encounters = pd.DataFrame([
    [101, 1, '2023-01-01', 'Annual checkup'],
    [102, 1, '2023-06-15', 'Follow-up'],
    [103, 2, '2023-03-11', 'Flu symptoms'],
    [104, 3, '2023-02-20', 'Blood work'],
    [105, 3, '2023-08-22', 'X-ray'],
    [106, 3, '2023-11-10', 'Consultation'],
    [107, 4, '2023-05-05', 'Injury treatment'],
    [108, 5, '2023-04-01', 'Routine exam'],
    [109, 5, '2023-12-19', 'Vaccination']
], columns=encounters_columns)

# MERGE
merged = pd.merge(encounters, patients, left_on='patient_id', right_on='id', suffixes=('_encounter', '_patient'))
pretty_print_dataframe(merged, msg="Merged")

# JOIN
# Only works on joining on indices, but does some more common sense things like not selecting the join key as a field
# Set index to 'id' for patients
patients_indexed = patients.set_index('id')

# Also set patient_id as index in encounters
encounters_indexed = encounters.set_index('patient_id')

joined = encounters_indexed.join(patients_indexed, how='left')
pretty_print_dataframe(joined, msg="Joined")

# CONCAT
# Add new records
patients_batch2 = pd.DataFrame([
    [6, 'Fay White', '1998-04-17'],
    [7, 'George Black', '1989-11-08']
], columns=patient_columns)

all_patients = pd.concat([patients, patients_batch2], ignore_index=True)
pretty_print_dataframe(all_patients, msg="Add Patients")



Merged


,id_encounter,patient_id,date,notes,id_patient,name,birthdate
0,101,1,2023-01-01,Annual checkup,1,Alice Smith,1990-05-01
1,102,1,2023-06-15,Follow-up,1,Alice Smith,1990-05-01
2,103,2,2023-03-11,Flu symptoms,2,Bob Jones,1985-07-13
3,104,3,2023-02-20,Blood work,3,Charlie Kim,1975-12-22
4,105,3,2023-08-22,X-ray,3,Charlie Kim,1975-12-22
5,106,3,2023-11-10,Consultation,3,Charlie Kim,1975-12-22
6,107,4,2023-05-05,Injury treatment,4,Dana Lee,1992-09-03
7,108,5,2023-04-01,Routine exam,5,Eli Brown,2000-01-30
8,109,5,2023-12-19,Vaccination,5,Eli Brown,2000-01-30


Joined


,id,date,notes,name,birthdate
1,101,2023-01-01,Annual checkup,Alice Smith,1990-05-01
1,102,2023-06-15,Follow-up,Alice Smith,1990-05-01
2,103,2023-03-11,Flu symptoms,Bob Jones,1985-07-13
3,104,2023-02-20,Blood work,Charlie Kim,1975-12-22
3,105,2023-08-22,X-ray,Charlie Kim,1975-12-22
3,106,2023-11-10,Consultation,Charlie Kim,1975-12-22
4,107,2023-05-05,Injury treatment,Dana Lee,1992-09-03
5,108,2023-04-01,Routine exam,Eli Brown,2000-01-30
5,109,2023-12-19,Vaccination,Eli Brown,2000-01-30


Add Patients


,id,name,birthdate
0,1,Alice Smith,1990-05-01
1,2,Bob Jones,1985-07-13
2,3,Charlie Kim,1975-12-22
3,4,Dana Lee,1992-09-03
4,5,Eli Brown,2000-01-30
5,6,Fay White,1998-04-17
6,7,George Black,1989-11-08


# Additional Concepts

## In-Place = False
Generally you will be creating new Dataframes as opposed to modifying existing ones

In some Pandas functions you can pass inplace=True and the Dataframe will be modified instead of generating a new one: This is generally discouraged and not done to avoid side effects in other parts of your ode that rely on the same DataFrame and because chaining methods is so common when using Dataframes, which can't be done with inplace=True because None is returned.

In [ ]:
# In General, modifications to dataframes return new dataframes, though inplace=True can be supplied in many cases # for more memory efficient operations if you don't need the initial dataframe anymore

# Here are some ways of filtering the dataframe on the condition population > 1000
# We will only show a couple columns
# SELECT households, population FROM df WHERE population > 1000
cols_to_show = ['households', 'population']

new_df = df[(df['population'] > 1000) & (df['households'] > 500)]
print(new_df[cols_to_show].head())

df.query('population > 1000 and households > 500', inplace=True)
print(df[cols_to_show].head(5))


## One index to rule them all
By default, an integer index is created for every Series and Dataframe. You can set a different column or columns but uniqueness is not necessarily enforced and some operations assume a unique index.

indices are:
  - immutable
  - should be unique, but not necessarily enforced if you override default

## Understanding Axis

Many Pandas functions have an optional axis argument that takes either None, 0 or 1, where 1=Columns and 0=Rows. It might seem counter-intuitive, so it is worth clarifying with a couple of examples.

In [ ]:
# Taking the mean of a Dataframe. Axis can be 0/index or 1/columns. Let's use the string to avoid confusion
# If we set the axis to be index, which is always at the row level, we get the aggregate per column
print(df.aggregate('sum', axis='index'))

# If we set the axis to columns we get the aggregate per row
print()
print(df[['housing_median_age', 'population']].head(1))
df[['housing_median_age', 'population']].aggregate('sum', axis='columns')

# One way to think about this is that the axis defines what is getting collapsed. So when we have axis=index, we are saying there should only be one returned value for all indices(which are rows). If axis=columns, then there should be only one column in the returned dataframe.

In [ ]:
# Pandas can be thought of as way to access a data table/2D matrix using labels
# and boolean conditions instead of indices.

t = [
    ['a', 'b', 'c'],
    [1,2,3],
    [4,5,6]
]

print(f"list of lists {t[1][2]}")

# Create a dataframe with n as columns and m as a record
df = pd.DataFrame(data=t[1:], columns=t[:1][0])

print(df.info())
# This is not the same as above because the column row is not part of the data
print(f"Dataframe position access {df.iloc[1,2]}")

print(f"Dataframe label and logic access \n{df.loc[df['c']==3,['c']]}")


In [ ]:
# 1. Input is a string: Return a single column
one = df['population']
two = df.__getitem__('population')
print(one.compare(two))

# OR
df.filter(items=['population'])

# 2. Input list of strings: To return a subset of columns when passed a list of strings
df[['population', 'households']]

# 3. Input is a mask(set of boolean conditions): To return a subset of data based on conditions
df[(df['population'] > 1000) & (df['households'] > 500)]

# 4. To return a subset of data based on applying a function
df[lambda df: df['population'] > 1000]

# These are included because you will see a lot of examples like this, but maybe it is best to just use the .loc method
# df.loc(row_conditions, column_conditions)
df.loc[(df['population'] > 1000) & (df['households'] > 500), ['population', 'households']]


# Add this more advanced exmaple to the examples library
# def high_corr_with_target(df, target="median_house_value", threshold=0.5):
#     numeric = df.select_dtypes("number")
#     corr = numeric.corr()[target].drop(target)
#     return corr[abs(corr) > threshold].index

# print(df[high_corr_with_target])

# def top_corr_column(df, target="score", threshold=0.5):
#     numeric = df.select_dtypes("number")

#     if target not in numeric.columns:
#         raise ValueError(f"Target column '{target}' is not numeric or not in DataFrame.")

#     corr_series = numeric.corr()[target].drop(target)
#     filtered = corr_series[abs(corr_series) > threshold]

#     if filtered.empty:
#         return None  # or raise / return default

#     top = filtered.abs().idxmax()
#     return top, corr_series[top]

# from functools import partial
# high_corr_age = partial(high_corr_with_target, target="median_house_value")

